# NB10 — 후보지 종합 점수 산정 (Candidate Site Scoring)

공영주차장 후보를 **버티포트·드론 배송 허브** 관점에서 종합 평가합니다.
NB11의 최적 거점 선정을 위한 사전 점수 준비 단계입니다.

## 입력 데이터

| 파일 | 역할 |
|------|------|
| `parking_candidates.gpkg` | 공영주차장 후보지 (NB06) |
| `delivery_zones.gpkg` | 배송 운영 구역 (NB09b) |
| `constraint_grid.gpkg` | H3 제약 레이어 (NB09) |
| `demand_grid.gpkg` | H3 수요 그리드 (NB08) |
| `corporate_context_dong.gpkg` | 기업 밀도 (NB07a, 선택) |

## 점수 구성

```
candidate_score_primary =
    0.35 × demand_context_score
  + 0.25 × constraint_context_score
  + 0.25 × parking_site_score
  + 0.10 × zone_score
  + 0.05 × corporate_context_score
```

공역 공식 데이터 미확보 시 `candidate_score_conservative`를 순위 기준으로 사용합니다.

In [1]:
import warnings
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
from datetime import datetime
from shapely.geometry import mapping
from shapely.ops import unary_union

warnings.filterwarnings('ignore')

_cwd = Path.cwd()
if _cwd.name == '02_analysis':
    BASE = _cwd.parent
elif (_cwd / '02_analysis').exists():
    BASE = _cwd
else:
    BASE = _cwd.parent
PROC = BASE / 'processed'

print(f'Project root : {BASE}')
print(f'Processed    : {PROC}')
print(f'Run time     : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')


Project root : C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam_reset
Processed    : C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam_reset\processed
Run time     : 2026-05-12 09:41:35


In [2]:
# ══════════════════════════════════════════════════════════
#  입력 파일 로드 및 스키마 검증
# ══════════════════════════════════════════════════════════

REQUIRED_FILES = {
    'parking_candidates.gpkg': 'NB06_parking_candidate_sites.ipynb',
    'constraint_grid.gpkg':    'NB09_constraint_layer_scoring.ipynb',
    'delivery_zones.gpkg':     'NB09b_delivery_zone_classification.ipynb',
    'demand_grid.gpkg':        'NB08_demand_grid_index.ipynb',
}
for fname, nb_ref in REQUIRED_FILES.items():
    p = PROC / fname
    if not p.exists():
        raise FileNotFoundError(f'{fname} 없음. {nb_ref} 먼저 실행하세요.')

# Load parking candidates
lots_raw = gpd.read_file(PROC / 'parking_candidates.gpkg')
print(f'parking_candidates : {len(lots_raw)} lots   CRS={lots_raw.crs}')
print(f'  columns: {list(lots_raw.columns)}')

# Load delivery zones
dz_raw = gpd.read_file(PROC / 'delivery_zones.gpkg')
print(f'delivery_zones     : {len(dz_raw)} cells   CRS={dz_raw.crs}')

# Load constraint grid (for local scores)
cg_raw = gpd.read_file(PROC / 'constraint_grid.gpkg')
print(f'constraint_grid    : {len(cg_raw)} cells')

# Load demand grid
dmd_raw = gpd.read_file(PROC / 'demand_grid.gpkg')
print(f'demand_grid        : {len(dmd_raw)} cells')

# Optional: corporate context
corp_path = PROC / 'corporate_context_dong.gpkg'
HAS_CORP = corp_path.exists()
if HAS_CORP:
    corp_raw = gpd.read_file(corp_path)
    print(f'corporate_context  : {len(corp_raw)} dongs   cols={list(corp_raw.columns)}')
    HAS_CORP = 'Ec' in corp_raw.columns
    if not HAS_CORP:
        print('  WARNING: Ec column not found in corporate_context — skipping')
else:
    print('corporate_context  : not found (optional — will use default 0.50)')

# ── column aliases ──────────────────────────────────────────────────────────
for df in [lots_raw, dz_raw, cg_raw, dmd_raw]:
    if 'h3_index' not in df.columns and 'h3_id' in df.columns:
        df['h3_index'] = df['h3_id']

# ── numeric coercion ─────────────────────────────────────────────────────────
def to_num(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

to_num(lots_raw, ['lat', 'lon', 'capacity_num', 'avg_monthly_entries',
                   'parking_candidate_score'])
to_num(dz_raw, ['Ds', 'drone_score', 'robot_score', 'service_priority_score',
                 'constraint_score', 'constraint_score_with_airspace_proxy'])
to_num(cg_raw, ['constraint_score', 'constraint_score_with_airspace_proxy',
                 'score_airspace', 'score_obstacle', 'score_terrain'])

# ── bool-string columns ──────────────────────────────────────────────────────
BOOL_MAP = {'True': True, 'False': False, True: True, False: False, 'nan': False, 'None': False}
for col in ['hard_exclusion_flag', 'airspace_proxy_flag', 'airspace_proxy_caution_flag', 'priority_zone']:
    if col in dz_raw.columns:
        dz_raw[col] = dz_raw[col].map(BOOL_MAP).fillna(False).astype(bool)

# Check for proxy columns
HAS_PROXY_DZ = 'zone_with_airspace_proxy' in dz_raw.columns
HAS_PROXY_CS = 'constraint_score_with_airspace_proxy' in dz_raw.columns
print(f'\n  HAS_PROXY_DZ: {HAS_PROXY_DZ}  HAS_PROXY_CS: {HAS_PROXY_CS}  HAS_CORP: {HAS_CORP}')

# Check airspace status
if 'airspace_data_status' in dz_raw.columns:
    ads = dz_raw['airspace_data_status'].dropna().unique()
    AIRSPACE_OFFICIAL_MISSING = any('proxy' in str(s) or 'official_missing' in str(s) for s in ads)
else:
    AIRSPACE_OFFICIAL_MISSING = True
print(f'  AIRSPACE_OFFICIAL_MISSING: {AIRSPACE_OFFICIAL_MISSING}')


parking_candidates : 172 lots   CRS=EPSG:4326
  columns: ['lot_id', 'lot_name', 'lot_class', 'lot_subtype', 'address_road', 'address_jibun', 'capacity_raw', 'op_days', 'op_wday_start', 'op_wday_end', 'op_sat_start', 'op_sat_end', 'op_hol_start', 'op_hol_end', 'fee_info', 'operator', 'lat', 'lon', 'has_disabled_spaces', 'data_date', 'valid_month_count', 'avg_monthly_entries', 'total_valid_entries', 'usage_data_available', 'capacity_num', 'capacity_score', 'turnover_per_space', 'availability_score', 'usage_confidence', 'subtype_score', 'op_hours', 'operation_score', 'parking_candidate_score', 'parking_candidate_score_confidence', 'candidate_rank', 'candidate_grade', 'candidate_reason', 'risk_note', 'geometry']
delivery_zones     : 1947 cells   CRS=EPSG:4326
constraint_grid    : 1947 cells
demand_grid        : 1947 cells


corporate_context  : 50 dongs   cols=['ADM_NM', 'GU_NM', 'CSV_ADMI_CD', 'area_km2', 'avg_total_corp', 'weighted_corp', 'corporate_density_per_km2', 'weighted_corp_density_per_km2', 'Ec', 'top_industry', 'total_merchant_count', 'delivery_related_merchant_count', 'delivery_merchant_ratio', 'Od_merchant', 'corporate_context_note', 'geometry']

  HAS_PROXY_DZ: False  HAS_PROXY_CS: False  HAS_CORP: True
  AIRSPACE_OFFICIAL_MISSING: False


In [3]:
# ══════════════════════════════════════════════════════════
#  주차장 → H3 셀 공간 매핑
# ══════════════════════════════════════════════════════════

# Work in EPSG:5179 for all spatial ops
lots_5179 = lots_raw.to_crs('EPSG:5179') if lots_raw.crs.to_epsg() != 5179 else lots_raw.copy()
dz_5179   = dz_raw.to_crs('EPSG:5179')   if dz_raw.crs.to_epsg() != 5179   else dz_raw.copy()
cg_5179   = cg_raw.to_crs('EPSG:5179')   if cg_raw.crs.to_epsg() != 5179   else cg_raw.copy()
dmd_5179  = dmd_raw.to_crs('EPSG:5179')  if dmd_raw.crs.to_epsg() != 5179  else dmd_raw.copy()

# Ensure lot_id column
if 'lot_id' not in lots_5179.columns:
    lots_5179['lot_id'] = [f'LOT_{i:04d}' for i in range(len(lots_5179))]
    lots_raw['lot_id']  = lots_5179['lot_id'].values

# Deduplicate on lot_id to prevent row expansion in downstream merges
n_before = len(lots_5179)
lots_5179 = lots_5179.drop_duplicates(subset=['lot_id']).reset_index(drop=True)
if len(lots_5179) < n_before:
    print(f'  WARNING: dropped {n_before - len(lots_5179)} duplicate lot_id rows')

# --- Step 1: assign each lot to local H3 cell (point-in-polygon) ---
LOCAL_H3_COLS = ['h3_index', 'ADM_NM', 'GU_NM', 'CSV_ADMI_CD',
                 'Ds', 'demand_grade', 'delivery_zone',
                 'drone_score', 'robot_score', 'service_priority_score',
                 'constraint_score', 'hard_exclusion_flag',
                 'airspace_proxy_caution_flag', 'priority_zone']
if HAS_PROXY_DZ:
    LOCAL_H3_COLS += ['zone_with_airspace_proxy']
if HAS_PROXY_CS:
    LOCAL_H3_COLS += ['constraint_score_with_airspace_proxy']
LOCAL_H3_COLS = [c for c in LOCAL_H3_COLS if c in dz_5179.columns]

dz_local = dz_5179[LOCAL_H3_COLS + ['geometry']].copy()

# Primary: point within polygon
lots_joined = gpd.sjoin(lots_5179, dz_local, how='left', predicate='within')
matched_mask = lots_joined['h3_index'].notna()
n_matched = matched_mask.sum()
print(f'Point-in-polygon match: {n_matched} / {len(lots_5179)} lots')

# Fallback: nearest H3 centroid for unmatched lots
if n_matched < len(lots_5179):
    unmatched_idx = lots_joined[~matched_mask].index
    print(f'  Unmatched: {len(unmatched_idx)} lots — using nearest H3 cell')

    h3_centroids_5179 = dz_5179.copy()
    h3_centroids_5179['geometry'] = dz_5179.geometry.centroid
    h3_cent_gdf = h3_centroids_5179[LOCAL_H3_COLS + ['geometry']]

    # nearest join
    unmatched_lots = lots_5179.loc[unmatched_idx]
    nearest_joined = gpd.sjoin_nearest(
        unmatched_lots, h3_cent_gdf,
        how='left', distance_col='nearest_h3_m'
    )
    # Patch into lots_joined
    for col in LOCAL_H3_COLS:
        if col in nearest_joined.columns:
            lots_joined.loc[unmatched_idx, col] = nearest_joined[col].values
    lots_joined.loc[unmatched_idx, 'nearest_h3_m'] = nearest_joined['nearest_h3_m'].values
else:
    lots_joined['nearest_h3_m'] = 0.0

# Clean up extra index columns
lots_joined = lots_joined.loc[~lots_joined.index.duplicated(keep='first')]
for col in ['index_right', 'index_right_x', 'index_right_y']:
    if col in lots_joined.columns:
        lots_joined.drop(columns=[col], inplace=True)

print(f'  Final: {len(lots_joined)} lots with local H3 context')
print(f'  Sample delivery_zone: {lots_joined["delivery_zone"].value_counts().head()}')


Point-in-polygon match: 171 / 171 lots
  Final: 171 lots with local H3 context
  Sample delivery_zone: delivery_zone
부적합      166
드론우선       4
하이브리드      1
Name: count, dtype: int64


In [4]:
# ══════════════════════════════════════════════════════════
#  1km 집수 구역(Catchment) — 버퍼 생성 및 H3 집계
# ══════════════════════════════════════════════════════════

CATCHMENT_M = 1000

# Buffer lot points
lot_bufs_5179 = lots_5179[['lot_id', 'geometry']].copy()
lot_bufs_5179['geometry'] = lots_5179.geometry.buffer(CATCHMENT_M)

# Spatial join: buffer intersects H3 cells
CATCH_COLS = [
    'h3_index', 'ADM_NM', 'GU_NM', 'Ds', 'demand_grade',
    'delivery_zone', 'service_priority_score', 'constraint_score',
    'drone_score', 'robot_score', 'priority_zone',
]
if HAS_PROXY_DZ:  CATCH_COLS += ['zone_with_airspace_proxy']
if HAS_PROXY_CS:  CATCH_COLS += ['constraint_score_with_airspace_proxy']
CATCH_COLS = [c for c in CATCH_COLS if c in dz_5179.columns]

dz_for_catch = dz_5179[CATCH_COLS + ['geometry']].reset_index(drop=True)

catchment_raw = gpd.sjoin(
    lot_bufs_5179,
    dz_for_catch,
    how='left',
    predicate='intersects'
)
catchment_raw.drop(columns=['index_right'], errors='ignore', inplace=True)
catchment_raw = catchment_raw[catchment_raw['h3_index'].notna()]
print(f'Catchment pairs (lot × H3): {len(catchment_raw):,}')

# ── Compute distance from lot centre to H3 centroid ──────
lot_pts_5179 = lots_5179[['lot_id', 'geometry']].copy()  # original points
h3_cents_5179 = dz_5179[['h3_index', 'geometry']].copy()
h3_cents_5179['geometry'] = h3_cents_5179.geometry.centroid

cov_dist = catchment_raw[['lot_id', 'h3_index']].copy()
cov_dist = cov_dist.merge(
    lot_pts_5179.rename(columns={'geometry': 'lot_pt'}),
    on='lot_id', how='left'
)
cov_dist = cov_dist.merge(
    h3_cents_5179.rename(columns={'geometry': 'h3_cent'}),
    on='h3_index', how='left'
)
cov_dist['distance_to_lot_m'] = [
    float(lp.distance(hc)) if (lp is not None and hc is not None) else np.nan
    for lp, hc in zip(cov_dist.get('lot_pt', [None]*len(cov_dist)),
                      cov_dist.get('h3_cent', [None]*len(cov_dist)))
]
catchment_raw['distance_to_lot_m'] = cov_dist['distance_to_lot_m'].values

# ── Aggregate per lot ────────────────────────────────────
ZONE_SCORE_MAP = {
    '하이브리드': 1.00, '드론우선': 0.85, '로봇우선': 0.75,
    '보완검토': 0.45, '부적합': 0.00,
}

def _mode(s):
    vc = s.dropna().value_counts()
    return vc.index[0] if len(vc) > 0 else np.nan

agg_dict = {
    'covered_cell_count':           ('h3_index', 'count'),
    'catchment_mean_Ds':            ('Ds',        'mean'),
    'catchment_total_Ds':           ('Ds',        'sum'),
    'catchment_max_Ds':             ('Ds',        'max'),
    'catchment_mean_service_priority': ('service_priority_score', 'mean'),
    'catchment_mean_constraint':    ('constraint_score', 'mean'),
    'catchment_mean_drone_score':   ('drone_score', 'mean'),
    'catchment_mean_robot_score':   ('robot_score', 'mean'),
}

def _agg_mode(df, lot_id_col='lot_id'):
    return df.groupby(lot_id_col)['delivery_zone'].agg(_mode).reset_index(name='catchment_primary_zone_mode')

catchment_stats = catchment_raw.groupby('lot_id').agg(**agg_dict).reset_index()
catchment_stats['covered_priority_cell_count'] = (
    catchment_raw[catchment_raw['priority_zone'] == True]
    .groupby('lot_id')['h3_index'].count()
    .reindex(catchment_stats['lot_id']).fillna(0).values
)
catchment_stats['covered_priority_ratio'] = (
    catchment_stats['covered_priority_cell_count'] / catchment_stats['covered_cell_count'].replace(0, 1)
)
zone_mode = _agg_mode(catchment_raw)
catchment_stats = catchment_stats.merge(zone_mode, on='lot_id', how='left')

if HAS_PROXY_DZ and 'zone_with_airspace_proxy' in catchment_raw.columns:
    proxy_mode = catchment_raw.groupby('lot_id')['zone_with_airspace_proxy'].agg(_mode).reset_index(name='catchment_proxy_zone_mode')
    catchment_stats = catchment_stats.merge(proxy_mode, on='lot_id', how='left')

if HAS_PROXY_CS and 'constraint_score_with_airspace_proxy' in catchment_raw.columns:
    proxy_cs = catchment_raw.groupby('lot_id')['constraint_score_with_airspace_proxy'].mean().reset_index(name='catchment_mean_proxy_constraint')
    catchment_stats = catchment_stats.merge(proxy_cs, on='lot_id', how='left')

print(f'Catchment stats: {len(catchment_stats)} lots')
print(f'  covered_cell_count: min={catchment_stats["covered_cell_count"].min():.0f}  '
      f'max={catchment_stats["covered_cell_count"].max():.0f}  '
      f'mean={catchment_stats["covered_cell_count"].mean():.1f}')
print(f'  covered_priority_ratio mean: {catchment_stats["covered_priority_ratio"].mean():.3f}')


Catchment pairs (lot × H3): 9,793


Catchment stats: 171 lots
  covered_cell_count: min=30  max=61  mean=57.3
  covered_priority_ratio mean: 0.021


In [5]:
# ══════════════════════════════════════════════════════════
#  H3 컨텍스트 병합 + 기업 컨텍스트 (선택)
# ══════════════════════════════════════════════════════════

# Merge catchment stats into lots
lots = lots_joined.merge(catchment_stats, on='lot_id', how='left')
print(f'After catchment merge: {len(lots)} lots')

# Normalize catchment_total_Ds with 1%–99% clip
p1  = lots['catchment_total_Ds'].quantile(0.01)
p99 = lots['catchment_total_Ds'].quantile(0.99)
if p99 > p1:
    lots['catchment_total_Ds_norm'] = (
        (lots['catchment_total_Ds'].clip(p1, p99) - p1) / (p99 - p1)
    ).clip(0, 1)
else:
    lots['catchment_total_Ds_norm'] = 0.5

# ── Optional corporate context ───────────────────────────
lots['corporate_context_missing'] = False
lots['corporate_context_score']   = 0.50   # default

if HAS_CORP:
    corp_df = pd.DataFrame(corp_raw[['CSV_ADMI_CD', 'ADM_NM', 'Ec']].drop_duplicates())
    corp_df['CSV_ADMI_CD'] = corp_df['CSV_ADMI_CD'].astype(str).str.zfill(8)

    # Try merge by CSV_ADMI_CD first (deduplicate to prevent row expansion)
    if 'CSV_ADMI_CD' in lots.columns:
        lots['_admi_cd'] = lots['CSV_ADMI_CD'].astype(str).str.zfill(8)
        corp_by_cd = (corp_df[['CSV_ADMI_CD', 'Ec']]
                      .drop_duplicates(subset=['CSV_ADMI_CD'])
                      .rename(columns={'CSV_ADMI_CD': '_admi_cd', 'Ec': '_Ec_cd'}))
        lots = lots.merge(corp_by_cd, on='_admi_cd', how='left')
        matched_cd = lots['_Ec_cd'].notna().sum()
        print(f'  Corp join by CSV_ADMI_CD: {matched_cd}/{len(lots)} matched')
    else:
        lots['_Ec_cd'] = np.nan

    # Fallback by ADM_NM (deduplicate to prevent row expansion)
    if 'ADM_NM' in lots.columns:
        corp_by_nm = (corp_df[['ADM_NM', 'Ec']]
                      .drop_duplicates(subset=['ADM_NM'])
                      .rename(columns={'Ec': '_Ec_nm'}))
        lots = lots.merge(corp_by_nm, on='ADM_NM', how='left')
    else:
        lots['_Ec_nm'] = np.nan

    lots['_Ec_final'] = lots['_Ec_cd'].fillna(lots.get('_Ec_nm', pd.Series(np.nan, index=lots.index)))
    n_still_missing = lots['_Ec_final'].isna().sum()
    if n_still_missing > 0:
        print(f'  Corp: {n_still_missing} lots without Ec — using default 0.50')
    lots['corporate_context_score'] = lots['_Ec_final'].clip(0, 1).fillna(0.50)
    lots['corporate_context_missing'] = lots['_Ec_final'].isna()
    lots.drop(columns=['_admi_cd', '_Ec_cd', '_Ec_nm', '_Ec_final'], errors='ignore', inplace=True)
    print(f'  corporate_context_score: mean={lots["corporate_context_score"].mean():.3f}')


After catchment merge: 171 lots
  Corp join by CSV_ADMI_CD: 171/171 matched
  corporate_context_score: mean=0.138


In [6]:
# ══════════════════════════════════════════════════════════
#  구성 점수 계산
# ══════════════════════════════════════════════════════════

ZONE_SCORE_MAP = {
    '하이브리드': 1.00, '드론우선': 0.85, '로봇우선': 0.75,
    '보완검토': 0.45,   '부적합': 0.00,
}

# 1. parking_site_score
lots['parking_site_score'] = pd.to_numeric(
    lots.get('parking_candidate_score', pd.Series(np.nan, index=lots.index)),
    errors='coerce'
).clip(0, 1).fillna(0.40)

# 2. demand_context_score
mean_Ds = lots['catchment_mean_Ds'].fillna(lots['Ds'].fillna(0))
lots['demand_context_score'] = (
    0.60 * mean_Ds.clip(0, 1)
  + 0.40 * lots['catchment_total_Ds_norm'].fillna(0)
).clip(0, 1)

# 3. constraint_context_score
# Prefer local H3 constraint_score; fallback to catchment mean
if 'constraint_score_x' in lots.columns:
    lots.rename(columns={'constraint_score_x': 'constraint_score',
                         'constraint_score_y': 'constraint_score_catchment'}, inplace=True)
elif 'constraint_score' not in lots.columns:
    lots['constraint_score'] = np.nan

lots['constraint_context_score'] = (
    lots['constraint_score']
    .fillna(lots.get('catchment_mean_constraint', pd.Series(np.nan, index=lots.index)))
    .fillna(0.60)
    .clip(0, 1)
)

# 4. zone_score — from local delivery_zone
def _zone_score(z):
    return ZONE_SCORE_MAP.get(str(z).strip(), 0.30)

lots['zone_score'] = lots['delivery_zone'].apply(_zone_score)

# 5. corporate_context_score — already computed in Cell 5

print('[Component scores]')
for col in ['parking_site_score', 'demand_context_score',
            'constraint_context_score', 'zone_score', 'corporate_context_score']:
    s = lots[col]
    print(f'  {col:<30} [{s.min():.3f}, {s.max():.3f}]  mean={s.mean():.3f}')


[Component scores]
  parking_site_score             [0.099, 0.910]  mean=0.515
  demand_context_score           [0.106, 0.714]  mean=0.391
  constraint_context_score       [0.471, 0.897]  mean=0.671
  zone_score                     [0.000, 1.000]  mean=0.026
  corporate_context_score        [0.000, 1.000]  mean=0.138


In [7]:
# ══════════════════════════════════════════════════════════
#  종합 후보 점수 (Primary + Conservative)
# ══════════════════════════════════════════════════════════

W_DMD  = 0.35
W_CON  = 0.25
W_PKG  = 0.25
W_ZON  = 0.10
W_CRP  = 0.05
assert abs(W_DMD + W_CON + W_PKG + W_ZON + W_CRP - 1.0) < 1e-9

lots['candidate_score_primary'] = (
    W_DMD * lots['demand_context_score']
  + W_CON * lots['constraint_context_score']
  + W_PKG * lots['parking_site_score']
  + W_ZON * lots['zone_score']
  + W_CRP * lots['corporate_context_score']
).clip(0, 1)

# Conservative (proxy airspace)
if HAS_PROXY_DZ and 'zone_with_airspace_proxy' in lots.columns:
    lots['proxy_zone_score'] = lots['zone_with_airspace_proxy'].apply(_zone_score)
else:
    lots['proxy_zone_score'] = lots['zone_score']

# proxy constraint context score
if 'constraint_score_with_airspace_proxy' in lots.columns:
    proxy_cs = (lots['constraint_score_with_airspace_proxy']
                .fillna(lots.get('catchment_mean_proxy_constraint',
                                  lots['constraint_context_score']))
                .clip(0, 1))
elif 'catchment_mean_proxy_constraint' in lots.columns:
    proxy_cs = lots['catchment_mean_proxy_constraint'].fillna(lots['constraint_context_score']).clip(0, 1)
else:
    proxy_cs = lots['constraint_context_score']

lots['proxy_constraint_context_score'] = proxy_cs

lots['candidate_score_conservative'] = (
    W_DMD * lots['demand_context_score']
  + W_CON * lots['proxy_constraint_context_score']
  + W_PKG * lots['parking_site_score']
  + W_ZON * lots['proxy_zone_score']
  + W_CRP * lots['corporate_context_score']
).clip(0, 1)

# Ranking score selection
lots['airspace_review_required'] = AIRSPACE_OFFICIAL_MISSING
if AIRSPACE_OFFICIAL_MISSING:
    lots['candidate_score_for_ranking'] = lots['candidate_score_conservative']
    print('  airspace_review_required = True 전체 셀')
    print('  candidate_score_for_ranking = candidate_score_conservative')
else:
    lots['candidate_score_for_ranking'] = lots['candidate_score_primary']
    print('  airspace_review_required = False')
    print('  candidate_score_for_ranking = candidate_score_primary')

print(f'\n  candidate_score_primary     : mean={lots["candidate_score_primary"].mean():.3f}  '
      f'[{lots["candidate_score_primary"].min():.3f}, {lots["candidate_score_primary"].max():.3f}]')
print(f'  candidate_score_conservative: mean={lots["candidate_score_conservative"].mean():.3f}  '
      f'[{lots["candidate_score_conservative"].min():.3f}, {lots["candidate_score_conservative"].max():.3f}]')
score_diff = (lots['candidate_score_primary'] - lots['candidate_score_conservative'])
print(f'  score diff (primary-conservative): mean={score_diff.mean():.3f}  '
      f'max={score_diff.max():.3f}  min={score_diff.min():.3f}')


  airspace_review_required = False
  candidate_score_for_ranking = candidate_score_primary

  candidate_score_primary     : mean=0.443  [0.249, 0.693]
  candidate_score_conservative: mean=0.443  [0.249, 0.693]
  score diff (primary-conservative): mean=0.000  max=0.000  min=0.000


In [8]:
# ══════════════════════════════════════════════════════════
#  순위 / 등급 / 리스크 노트 / 후보 사유
# ══════════════════════════════════════════════════════════

# Rank (1 = best)
lots = lots.sort_values('candidate_score_for_ranking', ascending=False).reset_index(drop=True)
lots['candidate_rank'] = range(1, len(lots) + 1)

# Percentile (higher score = higher percentile)
n = len(lots)
lots['candidate_percentile'] = ((n - lots['candidate_rank'] + 1) / n * 100).round(1)

# Grade A/B/C/D
q80 = lots['candidate_score_for_ranking'].quantile(0.80)
q50 = lots['candidate_score_for_ranking'].quantile(0.50)
q20 = lots['candidate_score_for_ranking'].quantile(0.20)

def _grade(s):
    if s >= q80: return 'A'
    if s >= q50: return 'B'
    if s >= q20: return 'C'
    return 'D'

lots['candidate_grade'] = lots['candidate_score_for_ranking'].apply(_grade)

# ── NB10 risk note ───────────────────────────────────────
proxy_caution = lots.get('airspace_proxy_caution_flag', pd.Series(False, index=lots.index))
if proxy_caution.dtype == object:
    proxy_caution = proxy_caution.map({'True': True, 'False': False}).fillna(False)

def _risk_note(row):
    flags = []
    if proxy_caution.get(row.name, False):
        flags.append('공역 프록시 주의')
    zwp = str(row.get('zone_with_airspace_proxy', '')).strip()
    if zwp == '부적합':
        flags.append('보수 시나리오 부적합')
    if row.get('constraint_context_score', 1) < 0.45:
        flags.append('제약점수 낮음')
    if row.get('demand_context_score', 1) < 0.30:
        flags.append('수요권역 낮음')
    if row.get('parking_site_score', 1) < 0.40:
        flags.append('주차 후보 점수 낮음')
    return ' | '.join(flags) if flags else '이상 없음'

lots['nb10_risk_note'] = lots.apply(_risk_note, axis=1)

# ── NB10 candidate reason ────────────────────────────────
def _reason(row):
    zone  = str(row.get('delivery_zone', '')).strip()
    dmd   = row.get('demand_context_score', 0)
    pkg   = row.get('parking_site_score', 0)
    arw   = row.get('airspace_review_required', False)

    if zone in ('하이브리드', '드론우선', '로봇우선') and dmd >= 0.55 and pkg >= 0.50:
        return f'수요권역 높음, {zone} 운영 가능, 주차장 적합도 높음'
    if pkg >= 0.55 and arw:
        return '주차장 적합도 높으나 공역 프록시 검토 필요'
    if dmd >= 0.40 and (zone in ('부적합',) or row.get('constraint_context_score', 1) < 0.45):
        return '수요는 있으나 제약/공역 리스크 있음'
    if zone in ('하이브리드', '드론우선', '로봇우선'):
        return f'{zone} 구역 — 추가 현장 검토 권장'
    return '종합 점수 낮음, 추가 검토 필요'

lots['nb10_candidate_reason'] = lots.apply(_reason, axis=1)

print(f'[Ranking]  q20={q20:.3f}  q50={q50:.3f}  q80={q80:.3f}')
print(lots['candidate_grade'].value_counts().sort_index().to_string())
print(f'\n  airspace_review_required: {lots["airspace_review_required"].sum()} lots')


[Ranking]  q20=0.382  q50=0.442  q80=0.503
candidate_grade
A    35
B    51
C    51
D    34

  airspace_review_required: 0 lots


In [9]:
# ══════════════════════════════════════════════════════════
#  진단 및 요약 (Diagnostics & Summary)
# ══════════════════════════════════════════════════════════

print('=' * 65)
print('NB10 후보지 점수 — 진단 요약')
print('=' * 65)

# Input row counts
print(f'\n[입력]')
print(f'  parking_candidates : {len(lots_raw)}')
print(f'  delivery_zones     : {len(dz_raw)}')
print(f'  final lots (after join): {len(lots)}')

# Catchment coverage
print(f'\n[집수 구역]')
print(f'  covered_cell_count: '
      f'min={lots["covered_cell_count"].min():.0f}  '
      f'max={lots["covered_cell_count"].max():.0f}  '
      f'mean={lots["covered_cell_count"].mean():.1f}')
n_no_catchment = (lots['covered_cell_count'].fillna(0) == 0).sum()
if n_no_catchment > 0:
    print(f'  WARNING: {n_no_catchment} lots with 0 covered cells')

# Score ranges
print(f'\n[점수 범위 — 모두 [0,1] 이어야 함]')
score_cols = ['parking_site_score', 'demand_context_score', 'constraint_context_score',
              'zone_score', 'corporate_context_score',
              'candidate_score_primary', 'candidate_score_conservative', 'candidate_score_for_ranking']
all_in_range = True
for c in score_cols:
    if c not in lots.columns: continue
    lo, hi = lots[c].min(), lots[c].max()
    ok = (lo >= -1e-9) and (hi <= 1 + 1e-9)
    if not ok: all_in_range = False
    print(f'  {c:<34} [{lo:.3f}, {hi:.3f}]  mean={lots[c].mean():.3f}{"" if ok else "  PROBLEM"}')
print(f'  All in [0,1]: {all_in_range}')

# Top 20 by score
print(f'\n[상위 20 후보지 (candidate_score_for_ranking)]')
top20_cols = ['lot_name', 'GU_NM', 'ADM_NM', 'candidate_rank', 'candidate_grade',
              'candidate_score_for_ranking', 'candidate_score_primary',
              'delivery_zone', 'demand_context_score', 'constraint_context_score']
top20_cols = [c for c in top20_cols if c in lots.columns]
print(lots.head(20)[top20_cols].round(3).to_string(index=False))

# Top 20 score difference
print(f'\n[Primary - Conservative 점수 차이 상위 20]')
diff_col = 'score_diff'
lots[diff_col] = (lots['candidate_score_primary'] - lots['candidate_score_conservative']).abs()
diff20 = lots.nlargest(20, diff_col)[['lot_name', 'GU_NM', 'candidate_score_primary',
                                       'candidate_score_conservative', diff_col,
                                       'delivery_zone']].round(3)
print(diff20.to_string(index=False))
lots.drop(columns=[diff_col], inplace=True)

# Distribution by GU_NM
print(f'\n[구별 후보지 분포]')
if 'GU_NM' in lots.columns:
    gu_pivot = (lots.groupby(['GU_NM', 'candidate_grade'])
                    .size().unstack(fill_value=0))
    gu_pivot['합계'] = gu_pivot.sum(axis=1)
    print(gu_pivot.to_string())

# Distribution by delivery_zone
print(f'\n[배송 구역 × 등급]')
if 'delivery_zone' in lots.columns:
    zone_grade = lots.groupby(['delivery_zone', 'candidate_grade']).size().unstack(fill_value=0)
    print(zone_grade.to_string())

# Grade A airspace warning
grade_a = lots[lots['candidate_grade'] == 'A']
if AIRSPACE_OFFICIAL_MISSING:
    print(f'\n  WARNING: 공식 공역 데이터 없음. candidate_score_conservative 기준으로 순위 산정.')
a_proxy_count = (grade_a['airspace_review_required'] == True).sum() if len(grade_a) > 0 else 0
if len(grade_a) > 0 and a_proxy_count / len(grade_a) > 0.5:
    print(f'  WARNING: A등급 {len(grade_a)}개 중 {a_proxy_count}개 ({a_proxy_count/len(grade_a)*100:.0f}%) '
          f'가 공역 검토 필요.')


NB10 후보지 점수 — 진단 요약

[입력]
  parking_candidates : 172
  delivery_zones     : 1947
  final lots (after join): 171

[집수 구역]
  covered_cell_count: min=30  max=61  mean=57.3

[점수 범위 — 모두 [0,1] 이어야 함]
  parking_site_score                 [0.099, 0.910]  mean=0.515
  demand_context_score               [0.106, 0.714]  mean=0.391
  constraint_context_score           [0.471, 0.897]  mean=0.671
  zone_score                         [0.000, 1.000]  mean=0.026
  corporate_context_score            [0.000, 1.000]  mean=0.138
  candidate_score_primary            [0.249, 0.693]  mean=0.443
  candidate_score_conservative       [0.249, 0.693]  mean=0.443
  candidate_score_for_ranking        [0.249, 0.693]  mean=0.443
  All in [0,1]: True

[상위 20 후보지 (candidate_score_for_ranking)]
    lot_name GU_NM ADM_NM  candidate_rank candidate_grade  candidate_score_for_ranking  candidate_score_primary delivery_zone  demand_context_score  constraint_context_score
       수내역환승   분당구   수내1동               1              

candidate_grade   A   B   C   D  합계
GU_NM                              
분당구              16   7   4   1  28
수정구              11  27   9   5  52
중원구               8  17  38  28  91

[배송 구역 × 등급]
candidate_grade   A   B   C   D
delivery_zone                  
드론우선              2   2   0   0
부적합              32  49  51  34
하이브리드             1   0   0   0


In [10]:
# ══════════════════════════════════════════════════════════
#  출력 저장
# ══════════════════════════════════════════════════════════

# Define final column order
BASE_COLS = [
    'lot_id', 'lot_name', 'lot_class', 'lot_subtype',
    'address_road', 'address_jibun',
    'lat', 'lon', 'ADM_NM', 'GU_NM', 'CSV_ADMI_CD', 'h3_index', 'nearest_h3_m',
    'capacity_num', 'avg_monthly_entries',
    'parking_candidate_score', 'parking_candidate_score_confidence',
    'parking_site_score',
    'Ds', 'demand_grade',
    'delivery_zone', 'zone_with_airspace_proxy',
    'constraint_score', 'constraint_score_with_airspace_proxy',
    'drone_score', 'robot_score', 'service_priority_score',
    'covered_cell_count', 'covered_priority_cell_count', 'covered_priority_ratio',
    'catchment_mean_Ds', 'catchment_total_Ds', 'catchment_max_Ds',
    'catchment_mean_service_priority',
    'demand_context_score', 'constraint_context_score',
    'zone_score', 'corporate_context_score',
    'candidate_score_primary', 'candidate_score_conservative',
    'candidate_score_for_ranking',
    'candidate_rank', 'candidate_percentile', 'candidate_grade',
    'airspace_review_required',
    'candidate_reason', 'risk_note',
    'nb10_candidate_reason', 'nb10_risk_note',
]
FINAL_COLS = [c for c in BASE_COLS if c in lots.columns]

# Rebuild GeoDataFrame — geometry is already in lots from the sjoin
if 'geometry' in lots.columns and lots['geometry'].notna().any():
    geom_crs = 'EPSG:5179'
    out_gdf = gpd.GeoDataFrame(
        lots[[c for c in FINAL_COLS if c in lots.columns] + ['geometry']],
        geometry='geometry', crs=geom_crs
    ).to_crs('EPSG:4326')
else:
    # Fallback: re-join geometry by lot_id
    lot_geoms = lots_5179[['lot_id', 'geometry']].drop_duplicates(subset=['lot_id'])
    lots_with_geom = lots[[c for c in FINAL_COLS if c in lots.columns]].merge(
        lot_geoms, on='lot_id', how='left'
    )
    out_gdf = gpd.GeoDataFrame(lots_with_geom, geometry='geometry', crs='EPSG:5179').to_crs('EPSG:4326')

# Cast types for GPKG
for col in out_gdf.select_dtypes(include=['bool']).columns:
    out_gdf[col] = out_gdf[col].astype(str)
for col in out_gdf.select_dtypes(include=['object']).columns:
    if col != 'geometry':
        out_gdf[col] = out_gdf[col].astype(str)
for col in ['gu_nm', 'adm_nm', 'h3_id']:
    if col in out_gdf.columns:
        out_gdf.drop(columns=[col], inplace=True)

# GPKG
gpkg_path = PROC / 'final_candidate_sites.gpkg'
out_gdf.to_file(gpkg_path, driver='GPKG')

# CSV
csv_path = PROC / 'final_candidate_sites.csv'
pd.DataFrame(out_gdf.drop(columns=['geometry'])).to_csv(csv_path, index=False, encoding='utf-8-sig')

# Candidate summary CSV
SUMMARY_COLS = [
    'lot_id', 'lot_name', 'GU_NM', 'ADM_NM',
    'candidate_rank', 'candidate_grade', 'candidate_score_for_ranking',
    'candidate_score_primary', 'candidate_score_conservative',
    'delivery_zone', 'demand_context_score', 'constraint_context_score',
    'parking_site_score', 'zone_score', 'corporate_context_score',
    'airspace_review_required', 'nb10_risk_note', 'nb10_candidate_reason',
]
summ_df = pd.DataFrame(out_gdf[[c for c in SUMMARY_COLS if c in out_gdf.columns]])
summ_path = PROC / 'candidate_site_summary.csv'
summ_df.to_csv(summ_path, index=False, encoding='utf-8-sig')

# Coverage parquet
PARQ_COLS_BASE = ['lot_id', 'lot_name', 'h3_index', 'ADM_NM', 'GU_NM',
                  'Ds', 'demand_grade', 'delivery_zone',
                  'service_priority_score', 'constraint_score', 'distance_to_lot_m']
if HAS_PROXY_DZ and 'zone_with_airspace_proxy' in catchment_raw.columns:
    PARQ_COLS_BASE.insert(PARQ_COLS_BASE.index('service_priority_score'),
                          'zone_with_airspace_proxy')

lot_names_map = lots[['lot_id', 'lot_name']].drop_duplicates().set_index('lot_id')['lot_name']
catchment_parq = catchment_raw[[c for c in PARQ_COLS_BASE if c in catchment_raw.columns]].copy()
if 'lot_name' not in catchment_parq.columns:
    catchment_parq['lot_name'] = catchment_parq['lot_id'].map(lot_names_map)

parq_path = PROC / 'candidate_h3_coverage.parquet'
try:
    catchment_parq.to_parquet(parq_path, index=False)
    print(f'  Saved: {parq_path.name}  ({parq_path.stat().st_size // 1024} KB)')
except Exception as e:
    parq_csv = PROC / 'candidate_h3_coverage.csv'
    catchment_parq.to_csv(parq_csv, index=False, encoding='utf-8-sig')
    print(f'  Parquet failed ({e}) — saved CSV: {parq_csv.name}')

for p in [gpkg_path, csv_path, summ_path]:
    print(f'  Saved: {p.name}  ({p.stat().st_size // 1024} KB)')


  Saved: candidate_h3_coverage.parquet  (178 KB)
  Saved: final_candidate_sites.gpkg  (224 KB)
  Saved: final_candidate_sites.csv  (122 KB)
  Saved: candidate_site_summary.csv  (45 KB)


In [11]:
# ══════════════════════════════════════════════════════════
#  지도 시각화 (Folium)
# ══════════════════════════════════════════════════════════
try:
    import folium

    GRADE_COLORS = {'A': '#1A5276', 'B': '#2ECC71', 'C': '#F39C12', 'D': '#95A5A6'}
    ZONE_FILL = {
        '하이브리드': '#27AE60', '드론우선': '#2980B9',
        '로봇우선': '#E67E22',  '보완검토': '#F1C40F', '부적합': '#95A5A6',
    }

    map_center = [out_gdf.geometry.y.mean(), out_gdf.geometry.x.mean()]
    fmap = folium.Map(location=map_center, zoom_start=12, tiles='CartoDB positron')

    # Optional: H3 delivery zone background (light, low opacity)
    dz_wgs84 = dz_raw.to_crs('EPSG:4326')
    if len(dz_wgs84) <= 500:
        for _, row in dz_wgs84.iterrows():
            zone = str(row.get('delivery_zone', ''))
            fill = ZONE_FILL.get(zone, '#cccccc')
            folium.GeoJson(
                mapping(row.geometry),
                style_function=lambda feat, f=fill: {
                    'fillColor': f, 'color': 'none',
                    'fillOpacity': 0.12, 'weight': 0,
                },
            ).add_to(fmap)

    # Candidate markers
    for _, row in out_gdf.iterrows():
        grade  = str(row.get('candidate_grade', 'D'))
        zone   = str(row.get('delivery_zone', ''))
        color  = GRADE_COLORS.get(grade, '#aaa')
        radius = max(4, min(14, float(row.get('capacity_num', 50) or 50) / 30))
        rnk    = row.get('candidate_rank', '?')
        scr    = float(row.get('candidate_score_for_ranking', 0))
        p_scr  = float(row.get('candidate_score_primary', 0))
        c_scr  = float(row.get('candidate_score_conservative', 0))
        dmd    = float(row.get('demand_context_score', 0))
        cst    = float(row.get('constraint_context_score', 0))
        pkg    = float(row.get('parking_site_score', 0))
        zwp    = str(row.get('zone_with_airspace_proxy', '-'))
        arw    = str(row.get('airspace_review_required', 'False'))
        risk   = str(row.get('nb10_risk_note', ''))
        name   = str(row.get('lot_name', ''))
        gu     = str(row.get('GU_NM', ''))
        adm    = str(row.get('ADM_NM', ''))

        tip = (f'<b>#{rnk} {name}</b><br>'
               f'{adm} ({gu})<br>'
               f'등급: <b>{grade}</b> | 구역: {zone}<br>'
               f'순위점수: {scr:.3f}<br>'
               f'주요: {p_scr:.3f} | 보수: {c_scr:.3f}<br>'
               f'수요: {dmd:.3f} | 제약: {cst:.3f} | 주차: {pkg:.3f}<br>'
               f'프록시구역: {zwp} | 공역검토: {arw}<br>'
               f'위험: {risk}')

        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=radius,
            color=color, fill=True, fill_color=color,
            fill_opacity=0.85, weight=1.5,
            tooltip=folium.Tooltip(tip, sticky=False),
        ).add_to(fmap)

    # Legend
    legend_html = '''
    <div style="position:fixed;bottom:30px;left:30px;z-index:1000;
                background:white;padding:10px;border-radius:8px;
                border:1px solid #ccc;font-size:12px;line-height:1.9">
    <b>후보지 등급</b><br>
    <span style="color:#1A5276;font-size:16px">&#9679;</span> A등급<br>
    <span style="color:#2ECC71;font-size:16px">&#9679;</span> B등급<br>
    <span style="color:#F39C12;font-size:16px">&#9679;</span> C등급<br>
    <span style="color:#95A5A6;font-size:16px">&#9679;</span> D등급<br>
    <br><b>배송 구역 (배경)</b><br>
    <span style="color:#27AE60">&#9632;</span> 하이브리드
    <span style="color:#2980B9">&#9632;</span> 드론우선<br>
    <span style="color:#E67E22">&#9632;</span> 로봇우선
    <span style="color:#F1C40F">&#9632;</span> 보완검토
    </div>
    '''
    fmap.get_root().html.add_child(folium.Element(legend_html))

    map_path = PROC / 'candidate_site_map.html'
    fmap.save(str(map_path))
    print(f'  Map saved: {map_path.name}  ({map_path.stat().st_size // 1024} KB)')
except Exception as e:
    print(f'  Map skipped: {e}')


  Map saved: candidate_site_map.html  (167 KB)


In [12]:
# ══════════════════════════════════════════════════════════
#  차트 시각화
# ══════════════════════════════════════════════════════════
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.font_manager as fm
    import matplotlib.patches as mpatches

    plt.rcParams['axes.unicode_minus'] = False
    for fname in ['Malgun Gothic', 'NanumGothic']:
        try:
            fm.findfont(fname, fallback_to_default=False)
            plt.rcParams['font.family'] = fname
            break
        except Exception:
            pass

    GRADE_COLORS = {'A': '#1A5276', 'B': '#2ECC71', 'C': '#F39C12', 'D': '#95A5A6'}

    fig = plt.figure(figsize=(18, 13))
    fig.suptitle('NB10 — 공영주차장 후보지 종합 점수 분석', fontsize=14, fontweight='bold')

    # ── Chart 1: Top 20 bar chart ────────────────────────
    ax1 = fig.add_subplot(2, 2, 1)
    top20 = lots.head(20).copy()
    bar_labels = [str(n)[:12] if pd.notna(n) else f'#{r}' for n, r in
                  zip(top20.get('lot_name', ['']*20), top20.get('candidate_rank', range(1, 21)))]
    bar_colors = [GRADE_COLORS.get(str(g), '#aaa') for g in top20.get('candidate_grade', ['D']*20)]
    ax1.barh(range(len(top20))[::-1], top20['candidate_score_for_ranking'],
             color=bar_colors, alpha=0.85, edgecolor='white')
    ax1.set_yticks(range(len(top20)))
    ax1.set_yticklabels([f'#{r} {l}' for r, l in
                          zip(top20['candidate_rank'], bar_labels)][::-1], fontsize=7)
    ax1.set_xlabel('candidate_score_for_ranking')
    ax1.set_title('상위 20 후보지')
    ax1.axvline(0.5, color='gray', ls='--', lw=0.8, alpha=0.5)
    ax1.grid(axis='x', alpha=0.3)
    patches = [mpatches.Patch(color=GRADE_COLORS[g], label=f'{g}등급') for g in 'ABCD']
    ax1.legend(handles=patches, fontsize=7, loc='lower right')

    # ── Chart 2: demand vs constraint scatter ────────────
    ax2 = fig.add_subplot(2, 2, 2)
    grad_arr = [str(g) for g in lots.get('candidate_grade', ['D']*len(lots))]
    for g, c in GRADE_COLORS.items():
        mask = [x == g for x in grad_arr]
        ax2.scatter(
            lots['demand_context_score'][mask],
            lots['constraint_context_score'][mask],
            c=c, label=f'{g}등급', alpha=0.7, s=25, edgecolors='none'
        )
    ax2.set_xlabel('demand_context_score')
    ax2.set_ylabel('constraint_context_score')
    ax2.set_title('수요 vs 제약 (등급별)')
    ax2.set_xlim(-0.02, 1.02); ax2.set_ylim(-0.02, 1.02)
    ax2.axvline(0.30, color='gray', ls=':', lw=0.8, alpha=0.5)
    ax2.axhline(0.45, color='gray', ls=':', lw=0.8, alpha=0.5)
    ax2.legend(fontsize=8); ax2.grid(alpha=0.25)

    # ── Chart 3: primary - conservative histogram ─────────
    ax3 = fig.add_subplot(2, 2, 3)
    diff = lots['candidate_score_primary'] - lots['candidate_score_conservative']
    ax3.hist(diff, bins=20, color='#8E44AD', alpha=0.8, edgecolor='white')
    ax3.axvline(0, color='black', ls='--', lw=1)
    ax3.axvline(diff.mean(), color='red', ls='--', lw=1, label=f'평균 {diff.mean():.3f}')
    ax3.set_xlabel('Primary − Conservative')
    ax3.set_ylabel('후보지 수')
    ax3.set_title('주요/보수 점수 차이 분포')
    ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

    # ── Chart 4: grade distribution by GU_NM ─────────────
    ax4 = fig.add_subplot(2, 2, 4)
    if 'GU_NM' in lots.columns:
        gu_grade = lots.groupby(['GU_NM', 'candidate_grade']).size().unstack(fill_value=0)
        gu_grade = gu_grade.reindex(columns=['A','B','C','D'], fill_value=0)
        gu_grade.plot(kind='bar', ax=ax4,
                      color=[GRADE_COLORS[g] for g in ['A','B','C','D']],
                      alpha=0.85, edgecolor='white')
        ax4.set_xticklabels(ax4.get_xticklabels(), rotation=0)
        ax4.set_xlabel('구')
        ax4.set_ylabel('후보지 수')
        ax4.set_title('구별 등급 분포')
        ax4.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    chart_path = PROC / 'candidate_site_chart.png'
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    print(f'  Chart saved: {chart_path.name}')
except Exception as e:
    print(f'  Chart skipped: {e}')


  Chart saved: candidate_site_chart.png


In [13]:
# ══════════════════════════════════════════════════════════
#  최종 검증
# ══════════════════════════════════════════════════════════
import geopandas as _gpd

print('=' * 65)
print('NB10 최종 검증')
print('=' * 65)

# Row count check
pkg_orig = len(_gpd.read_file(PROC / 'parking_candidates.gpkg'))
print(f'\n  parking_candidates원본  : {pkg_orig}')
print(f'  final_candidate_sites   : {len(out_gdf)}')
print(f'  Row match: {pkg_orig == len(out_gdf)}')

# Null checks
print('\n  [Null check]')
NULL_COLS = ['lot_id', 'candidate_rank', 'candidate_grade',
             'candidate_score_for_ranking', 'delivery_zone']
for c in NULL_COLS:
    if c not in out_gdf.columns:
        print(f'    {c:<28} COLUMN MISSING')
        continue
    n = (out_gdf[c].astype(str) == 'nan').sum()
    flag = '  <-- PROBLEM' if n > 0 else ''
    print(f'    {c:<28} {n} nulls{flag}')

null_geom = out_gdf.geometry.isna().sum()
print(f'    geometry                     {null_geom} nulls' + ('' if null_geom == 0 else '  <-- PROBLEM'))

# Score bounds
print('\n  [Score bounds]')
CHECK_SCORES = ['parking_site_score', 'demand_context_score', 'constraint_context_score',
                'zone_score', 'candidate_score_primary', 'candidate_score_conservative',
                'candidate_score_for_ranking']
for sc in CHECK_SCORES:
    if sc not in lots.columns: continue
    lo, hi = float(lots[sc].min()), float(lots[sc].max())
    ok = (lo >= -1e-9) and (hi <= 1 + 1e-9)
    print(f'    {sc:<34} [{lo:.3f}, {hi:.3f}]{"  OK" if ok else "  PROBLEM"}')

# Rank uniqueness
rank_unique = lots['candidate_rank'].nunique() == len(lots)
print(f'\n  candidate_rank unique: {rank_unique}')

# Output files
print('\n  [Output files]')
out_files = ['final_candidate_sites.gpkg', 'final_candidate_sites.csv',
             'candidate_site_summary.csv', 'candidate_site_map.html',
             'candidate_site_chart.png']
parq = PROC / 'candidate_h3_coverage.parquet'
csv_fallback = PROC / 'candidate_h3_coverage.csv'
if parq.exists(): out_files.append('candidate_h3_coverage.parquet')
elif csv_fallback.exists(): out_files.append('candidate_h3_coverage.csv')
else: out_files.append('candidate_h3_coverage.parquet (MISSING)')

for f in out_files:
    p = PROC / f
    if p.exists():
        print(f'    OK  {f}  ({p.stat().st_size // 1024} KB)')
    else:
        print(f'    MISSING  {f}')

print('\n✅ NB10 완료 — final_candidate_sites.gpkg / .csv 저장')
print('   NB11에서 최적 거점 k개 선정 예정')


NB10 최종 검증

  parking_candidates원본  : 172
  final_candidate_sites   : 171
  Row match: False

  [Null check]
    lot_id                       0 nulls
    candidate_rank               0 nulls
    candidate_grade              0 nulls
    candidate_score_for_ranking  0 nulls
    delivery_zone                0 nulls
    geometry                     0 nulls

  [Score bounds]
    parking_site_score                 [0.099, 0.910]  OK
    demand_context_score               [0.106, 0.714]  OK
    constraint_context_score           [0.471, 0.897]  OK
    zone_score                         [0.000, 1.000]  OK
    candidate_score_primary            [0.249, 0.693]  OK
    candidate_score_conservative       [0.249, 0.693]  OK
    candidate_score_for_ranking        [0.249, 0.693]  OK

  candidate_rank unique: True

  [Output files]
    OK  final_candidate_sites.gpkg  (224 KB)
    OK  final_candidate_sites.csv  (122 KB)
    OK  candidate_site_summary.csv  (45 KB)
    OK  candidate_site_map.html  (167 K